# 股價情緒分析 Pipeline（整合版）

Big Data & Business Analytics, National Taiwan University

## Pipeline 架構

| Step | 功能 | 主要輸出 |
|------|------|---------|
| 0 | 總參數區與共用工具 | `DB_CONFIG`, utils |
| 1 | 從 `stock_text` 篩選公司相關文章並去噪 | `articles_raw.csv` |
| 2 | 依 D+n 交易日漲跌幅貼上 1 / -1 / 0 標籤 | `articles_labeled.csv` |
| 3 | 斷詞、詞彙篩選、TF-IDF 向量化 | `articles_processed.csv`, `limit_model.npz` |
| 4 | 訓練 5 種分類器並評估 | 混淆矩陣、準確率 |
| 5 | 移動回測（真實預測情境） | 逐日預測明細、回測混淆矩陣 |

## 方法切換參數（method switches）

| 參數 | 選項 | 說明 |
|------|------|------|
| `TERM_SELECT_METHOD` | `'tf'` / `'chi2'` / `'sklearn'` | Step 3 詞彙選取方法 |
| `TOKENIZE_METHOD` | `'monpa'` / `'ngram'` | Step 3 / Step 5 斷詞方法 |
| `SPLIT_METHOD` | `'time'` / `'random'` | Step 4 訓練/測試切割方式 |

---
# Step 0 — 總參數區與共用工具

所有可調參數集中於此；各 Step 開頭若有覆寫需求，可在該 Step 的參數覆寫格中重新指派。

In [1]:
# ═══════════════ 總參數區 ═══════════════

# ── 目標公司 ──
COMPANY_ID   = '1519'
COMPANY_NAME = '華城'

# ── Step 1：篩選關鍵字 ──
INCLUDE_KEYWORDS = [
    '華城', '華城電機', '華城電能', 'EVALUE',
    'Fortune Electric', '重電三雄', '重電四君子'
]
EXCLUDE_KEYWORDS = [
    '京華城', '柯文哲', '橘子', '許芷瑜',
    '沈慶京', '應曉薇', '容積率', '弊案', '羈押'
]

# ── Step 2：貼標參數 ──
N_DAYS = 3       # D+n 交易日後的漲跌
SIGMA  = 0.03    # 漲跌門檻（3%）

# ── Step 3：斷詞與向量化 ──
TOKENIZE_METHOD    = 'monpa'   # 'monpa' | 'ngram'
TERM_SELECT_METHOD = 'sklearn' # 'tf' | 'chi2' | 'sklearn'
TOP_K_WORDS        = 200       # 看漲/看跌各取 K 個（sklearn 模式則為 K*2 總特徵數）
NGRAM_RANGE        = (1, 2)    # 詞級 N-gram 範圍（1=unigram, 2=bigram, 3=trigram）
                               # 先以 monpa 斷成詞，再由 TF-IDF 做詞組合
NGRAM_MIN          = 2         # TOKENIZE_METHOD='ngram' 時使用的字元 N-gram 最小長度
NGRAM_MAX          = 4         # TOKENIZE_METHOD='ngram' 時使用的字元 N-gram 最大長度

# ── Step 3 / 4：資料切分（Step 3 完成切分，Step 4 直接讀入）──
SPLIT_METHOD = 'random'    # 'time'（依時序切）| 'random'（stratified 隨機切）
TEST_RATIO   = 0.2
RANDOM_SEED  = 42

# ── Step 5：移動回測 ──
WINDOW_DAYS  = 120   # 訓練窗口（日曆天）
MIN_ARTICLES = 3     # 當日文章篇數門檻（低於此不出手）

# ── 檔名輸出 ──
PATH_ARTICLES_RAW       = 'articles_raw.csv'
PATH_ARTICLES_FILTERED  = 'articles_filtered_out.csv'
PATH_ARTICLES_LABELED   = 'articles_labeled.csv'
PATH_ARTICLES_PROCESSED = 'articles_processed.csv'
PATH_VECTOR_NPZ         = 'limit_model.npz'
PATH_FEATURE_NAMES      = 'feature_names.pkl'
PATH_X_TRAIN            = 'X_train.npz'
PATH_X_TEST             = 'X_test.npz'
PATH_Y_TRAIN            = 'y_train.npy'
PATH_Y_TEST             = 'y_test.npy'
PATH_BULL_TERMS         = f'top_{TOP_K_WORDS}_bull_terms.csv'
PATH_BEAR_TERMS         = f'top_{TOP_K_WORDS}_bear_terms.csv'
PATH_KEYWORDS_ANALYSIS  = 'keywords_analysis.csv'

In [58]:
# ── 資料庫連線設定（讀 .env）──
import os
from dotenv import load_dotenv

load_dotenv()

DB_CONFIG = {
    'host':     os.getenv('DB_HOST',     'localhost'),
    'user':     os.getenv('DB_USER',     'root'),
    'password': os.getenv('DB_PASSWORD', ''),
    'database': os.getenv('DB_NAME',     'bda2026'),
    'charset':  os.getenv('DB_CHARSET',  'utf8mb4'),
}

In [82]:
# %pip install stopwordsiso
# %pip install TCSP
# %pip install opencc
%pip install xgboost
%pip install lightgbm


   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
    --------------------------------------- 2.4/101.7 MB 19.1 MB/s eta 0:00:06
   ------ --------------------------------- 15.7/101.7 MB 49.4 MB/s eta 0:00:02
   ---------- ----------------------------- 27.5/101.7 MB 51.2 MB/s eta 0:00:02
   ----------------- ---------------------- 45.1/101.7 MB 59.8 MB/s eta 0:00:01
   ----------------------- ---------------- 60.3/101.7 MB 62.0 MB/s eta 0:00:01
   ---------------------------- ----------- 72.1/101.7 MB 60.5 MB/s eta 0:00:01
   ---------------------------------- ----- 87.6/101.7 MB 62.0 MB/s eta 0:00:01
   --------------------------------------  101.2/101.7 MB 62.1 MB/s eta 0:00:01
   --------------------------------------- 101.7/101.7 MB 56.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 1.5/1.5 MB 12.7 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated package

In [2]:
# ── 共用工具函式 ──
import re
import bisect
import csv
from collections import Counter, defaultdict
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import pymysql
from scipy import sparse


def db_connect():
    """建立 MySQL 連線。"""
    return pymysql.connect(**DB_CONFIG)


def is_valid_token(token: str) -> bool:
    """過濾 URL、純英數、雜訊 token。中文字需佔 80% 以上。"""
    if not token or len(token) < 2:
        return False
    if re.search(r'https?://', token):
        return False
    if re.search(r'www\.', token):
        return False
    chinese_chars = len(re.findall(r'[\u4e00-\u9fff]', token))
    if chinese_chars == 0:
        return False
    if chinese_chars / len(token) < 0.8:
        return False
    return True


# ── 三層停用詞 ──
# 第一層：stopwordsiso 繁中通用停用詞
# 第二層：TCSP 繁中停用詞表
from stopwordsiso import stopwords as iso_stopwords
from TCSP import read_stopwords_list
import requests
from opencc import OpenCC

BASE_STOPWORDS = set(iso_stopwords("zh")) | set(read_stopwords_list())


# 第三層：線上開源停用詞表 (Baidu, HIT, SCU)
print("正在從 GitHub 下載並轉換線上停用詞表（這會花個幾秒鐘）...")
cc = OpenCC('s2twp') # 初始化簡轉繁（台灣）
urls = {
    "baidu": "https://raw.githubusercontent.com/goto456/stopwords/master/baidu_stopwords.txt",
    "hit":   "https://raw.githubusercontent.com/goto456/stopwords/master/hit_stopwords.txt",
    "scu":   "https://raw.githubusercontent.com/goto456/stopwords/master/scu_stopwords.txt",
}

goto456_words = set()
for name, u in urls.items():
    try:
        response = requests.get(u, timeout=10) # 加上 timeout 避免卡死
        response.raise_for_status()
        for w in response.text.splitlines():
            w = w.strip()
            if w:
                goto456_words.add(cc.convert(w))
    except Exception as e:
        print(f"⚠️ 警告：無法下載 {name} 停用詞表，先略過！({e})")

# 第四層：手動補充停用詞
CUSTOM_STOPWORDS = {
    # === PTT 稱謂 ===
    '小弟', '大大', '鄉民', '版友', '各位', '請問', '樓主', '推主',
    '大家', '大家好', '您好', '謝謝', '感謝', '不客氣',
    # PTT 回文樣板
    '引述', '看板', '批踢踢', '發信站', '轉錄至', 'Re', 'Fw', '※',
    # 新聞樣板
    '記者', '報導', '撰文', '原文連結', '來源', '全文', '摘要',
    '本報訊', '綜合報導', '資料來源', '圖片來源', '編輯',
    # 填充語
    '覺得', '應該', '好像', '似乎', '其實', '這樣', '然後',
    '就是', '所以', '但是', '不過', '雖然', '因為', '如果',
    '可以', '可能', '已經', '一直', '真的', '還是', '還有',
    '一下', '一些', '一樣', '這個', '那個', '什麼', '怎麼',
    # 口語雜訊
    '哈哈', '哈哈哈', 'XD', 'xd', 'QQ', '嗚嗚', '欸', '啊', '喔', '哦', '呵', '嗯',
    # HTML 殘渣
    '閱讀全文', '繼續閱讀', '相關新聞', '延伸閱讀', '廣告', '更多內容', '訂閱',
    # 時間詞
    '今天', '昨天', '明天', '今日', '昨日', '明日',
    '上週', '下週', '本週', '這週', '上個月', '下個月', '今年', '去年', '明年',
    # 財經樣板
    '表示', '指出', '認為', '說明', '顯示', '指稱',
    '根據', '依據', '根本', '目前', '近期', '最近',
    # === 基本虛詞 ===
    '的', '了', '是', '在', '有', '和', '與', '或', '但', '也', '都', '就',
    '這', '那', '我', '你', '他', '她', '它', '們',
    # 代名詞 / 人稱
    '我們', '你們', '他們',
    # 連接詞 / 語氣詞
    '以及', '同時', '此外', '另外', '然而', '即使', '儘管',
    '無論', '不管', '只要', '只有', '除了', '除非',
    '假如', '要是', '既然', '因此', '於是', '進而', '從而',
    # 介詞
    '關於', '對於', '來自', '由於', '透過', '通過', '藉由', '基於', '針對',
    # 動詞樣板（無鑑別力）
    '提出', '提供', '進行', '包括', '需要', '帶來', '造成', '導致', '預計',
    # 其他
    '一個', '不是', '很多', '如何', '非常', '相關',
    '其中', '其他', '之前', '之後', '方面', '時間',
}

# ⚡ 合併四層停用詞 ⚡
STOPWORDS = BASE_STOPWORDS | goto456_words | CUSTOM_STOPWORDS
print(f"✅ 停用詞總數升級完成！現在有：{len(STOPWORDS)} 個")


def get_ngrams(text: str, min_n: int = NGRAM_MIN, max_n: int = NGRAM_MAX) -> list:
    """N-gram 字元切割：只保留中/英/數，去空白與純標點。"""
    clean_text = ''.join(re.findall(r'[\u4e00-\u9fffA-Za-z0-9]+', text))
    ngrams = []
    for n in range(min_n, max_n + 1):
        for i in range(len(clean_text) - n + 1):
            gram = clean_text[i:i + n]
            if re.search(r'[\u4e00-\u9fffA-Za-z]', gram):
                ngrams.append(gram)
    return ngrams


def tokenize(text: str, method: str = None) -> list:
    """依 TOKENIZE_METHOD 切割文字並過濾雜訊 token 與停用詞（四層）。"""
    method = method or TOKENIZE_METHOD

    if method == 'monpa':
        import monpa
        from monpa import utils as monpa_utils
        tokens = []
        for sent in monpa_utils.short_sentence(text):
            for term in monpa.cut(sent):
                term = term.strip()
                if len(term) > 1:
                    tokens.append(term)
    elif method == 'ngram':
        tokens = get_ngrams(text)
    else:
        raise ValueError(f'未知的 TOKENIZE_METHOD: {method}')

    return [t for t in tokens if is_valid_token(t) and t not in STOPWORDS]


print('共用工具已載入（含四層停用詞）')
print(f'停用詞總數：{len(STOPWORDS)} 個')

正在從 GitHub 下載並轉換線上停用詞表（這會花個幾秒鐘）...
✅ 停用詞總數升級完成！現在有：2785 個
共用工具已載入（含四層停用詞）
停用詞總數：2785 個


---
# Step 1 — 篩選相關文章

從 `stock_text` 以公司關鍵字擷取主文，去除雜訊（京華城弊案）、排行榜、數字比例過高的文章。

In [60]:
# ── Step 1 參數覆寫（如需）──
# 例：INCLUDE_KEYWORDS = [...]

# 取得股價資料的時間範圍
conn = db_connect()
cursor = conn.cursor()

# 先抓取資料庫中的最大與最小日期作為備用
cursor.execute(
    'SELECT MIN(trade_date), MAX(trade_date) FROM stock_prices WHERE company_id = %s',
    (COMPANY_ID,)
)
db_min_date, db_max_date = cursor.fetchone()

# 👇 判斷邏輯：如果有設定 START_DATE / END_DATE 就用設定的，否則用資料庫的
min_date = START_DATE if START_DATE else db_min_date
max_date = END_DATE if END_DATE else db_max_date

print(f'股價資料範圍設定為：{min_date} ～ {max_date}')

# 動態組合 SQL：包含條件 + 排除條件 + 時間範圍
search_keywords = INCLUDE_KEYWORDS + [COMPANY_ID]

include_clause = ' OR '.join(['(title LIKE %s OR content LIKE %s)'] * len(search_keywords))
include_params = [p for kw in search_keywords for p in (f'%{kw}%', f'%{kw}%')]

if EXCLUDE_KEYWORDS:
    exclude_clause = ' AND ' + ' AND '.join(
        ['(title NOT LIKE %s AND content NOT LIKE %s)'] * len(EXCLUDE_KEYWORDS)
    )
    exclude_params = [p for kw in EXCLUDE_KEYWORDS for p in (f'%{kw}%', f'%{kw}%')]
else:
    exclude_clause, exclude_params = '', []

sql_query = f"""
    SELECT no, post_time, title, content, s_name
    FROM stock_text
    WHERE ({include_clause})
      {exclude_clause}
      AND (content_type = 'main' OR content_type IS NULL)
      AND DATE(post_time) >= %s
      AND DATE(post_time) <= %s
    ORDER BY post_time
"""

final_params = include_params + exclude_params + [min_date, max_date]

print('正在進行深度篩選與去噪...')
cursor.execute(sql_query, final_params)
rows = cursor.fetchall()
print(f'關鍵字精確篩選後：{len(rows)} 篇')

cursor.close()
conn.close()

股價資料範圍設定為：2024-01-01 ～ 2025-02-27
正在進行深度篩選與去噪...
關鍵字精確篩選後：2447 篇


In [61]:
# ── 去除 boilerplate：排行榜、數字比例過高、太短的文章 ──
BOILERPLATE_TITLES = ['買賣超排行', '投信買賣', '外資買賣', '主力買賣', '法人買賣']


def is_boilerplate(title: str, content: str) -> bool:
    text = (title or '') + (content or '')
    if any(p in (title or '') for p in BOILERPLATE_TITLES):
        return True
    if len(text) == 0:
        return True
    if sum(c.isdigit() for c in text) / len(text) > 0.3:
        return True
    if len(text) < 50:
        return True
    return False


# rows 欄位順序：no, post_time, title, content, s_name
rows_kept    = [r for r in rows if not is_boilerplate(r[2], r[3])]
rows_removed = [r for r in rows if     is_boilerplate(r[2], r[3])]

print(f'篩選前：{len(rows)} 篇')
print(f'過濾掉：{len(rows_removed)} 篇')
print(f'保留：  {len(rows_kept)} 篇')

篩選前：2447 篇
過濾掉：354 篇
保留：  2093 篇


In [62]:
# ── 儲存 CSV ──
HEADER = ['no', 'post_time', 'title', 'content', 's_name']

for path, data, label in [
    (PATH_ARTICLES_RAW,      rows_kept,    '保留'),
    (PATH_ARTICLES_FILTERED, rows_removed, '過濾'),
]:
    with open(path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(HEADER)
        writer.writerows(data)
    print(f'已儲存 {path}（{label}）：{len(data)} 筆')

# 各平台分布統計
for data, label in [(rows_kept, '保留'), (rows_removed, '過濾')]:
    print(f'\n===== {label}文章 - 各平台分布（{len(data)} 筆）=====')
    for src, cnt in Counter(r[4] for r in data).most_common():
        print(f'  {src}: {cnt} 篇')

已儲存 articles_raw.csv（保留）：2093 筆
已儲存 articles_filtered_out.csv（過濾）：354 筆

===== 保留文章 - 各平台分布（2093 筆）=====
  Yahoo股市: 1469 篇
  Ptt: 239 篇
  Yahoo新聞: 204 篇
  Dcard: 178 篇
  Mobile01: 3 篇

===== 過濾文章 - 各平台分布（354 筆）=====
  Ptt: 240 篇
  Yahoo股市: 86 篇
  Dcard: 15 篇
  Yahoo新聞: 13 篇


---
# Step 2 — 貼股價標籤

對每篇文章，取其發文日（或下一個交易日）的收盤價，與 D+n 交易日後收盤價比較：
- 漲幅 > σ → **+1（看漲）**
- 跌幅 < -σ → **-1（看跌）**
- 否則 → **0（中性，不納入訓練）**

In [63]:
# ── Step 2 參數覆寫 ──
# 例：N_DAYS = 5; SIGMA = 0.02

# 讀該公司所有交易日與收盤價
conn = db_connect()
cursor = conn.cursor()
cursor.execute(
    '''
    SELECT trade_date, closing_price
    FROM stock_prices
    WHERE company_id = %s
    ORDER BY trade_date
    ''',
    (COMPANY_ID,)
)
price_rows = cursor.fetchall()
cursor.close()
conn.close()

trade_dates = [row[0] for row in price_rows]
price_dict  = {row[0]: float(row[1]) for row in price_rows}

print(f'共 {len(trade_dates)} 個交易日')
print(f'範圍：{trade_dates[0]} ～ {trade_dates[-1]}')

共 483 個交易日
範圍：2023-03-01 ～ 2025-02-27


In [64]:
# ── 交易日查詢輔助函式 ──
# [METHOD] 目前方法：固定 n 個交易日後 | 備用：事件窗口法（取最大漲跌）

def get_price_on_or_after(date, trade_dates, price_dict):
    """date 當天或之後第一個交易日的收盤價"""
    idx = bisect.bisect_left(trade_dates, date)
    if idx >= len(trade_dates):
        return None
    return price_dict[trade_dates[idx]]


def get_nth_trading_day_price(date, n, trade_dates, price_dict):
    """date 之後第 n 個交易日（不含當日）的收盤價"""
    idx = bisect.bisect_right(trade_dates, date)
    target_idx = idx + n - 1
    if target_idx >= len(trade_dates):
        return None
    return price_dict[trade_dates[target_idx]]

In [65]:
# ── 對每篇文章貼標籤 ──
articles = []
with open(PATH_ARTICLES_RAW, newline='', encoding='utf-8') as f:
    articles = list(csv.DictReader(f))
print(f'讀入 {len(articles)} 篇文章')

labeled = []
skip_count = 0

for art in articles:
    try:
        post_date = datetime.strptime(art['post_time'], '%Y-%m-%d %H:%M:%S').date()
    except Exception:
        skip_count += 1
        continue

    price_d  = get_price_on_or_after(post_date, trade_dates, price_dict)
    price_dn = get_nth_trading_day_price(post_date, N_DAYS, trade_dates, price_dict)

    if price_d is None or price_dn is None or price_d == 0:
        skip_count += 1
        continue

    pct = (price_dn - price_d) / price_d

    if pct > SIGMA:
        label = 1
    elif pct < -SIGMA:
        label = -1
    else:
        label = 0

    art['label']      = label
    art['pct_change'] = round(pct * 100, 2)
    labeled.append(art)

print(f'貼標完成：{len(labeled)} 篇（跳過 {skip_count} 篇）')

# 儲存
fieldnames = ['no', 'post_time', 'title', 'content', 's_name', 'label', 'pct_change']
with open(PATH_ARTICLES_LABELED, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for art in labeled:
        writer.writerow({k: art.get(k, '') for k in fieldnames})

# 標籤分布
label_cnt = Counter(art['label'] for art in labeled)
print(f'看漲 ( 1)：{label_cnt[ 1]} 篇')
print(f'看跌 (-1)：{label_cnt[-1]} 篇')
print(f'中性 ( 0)：{label_cnt[ 0]} 篇（不納入訓練）')
print(f'已儲存至 {PATH_ARTICLES_LABELED}')

讀入 2093 篇文章
貼標完成：2084 篇（跳過 9 篇）
看漲 ( 1)：685 篇
看跌 (-1)：567 篇
中性 ( 0)：832 篇（不納入訓練）
已儲存至 articles_labeled.csv


---
# Step 3 — 斷詞、詞彙篩選、向量化

## 方法切換

- **`TOKENIZE_METHOD`**：
  - `'monpa'`：繁中分詞器（較慢，語意較精準）
  - `'ngram'`：字元 N-gram（快，無語言知識依賴）

- **`TERM_SELECT_METHOD`**：
  - `'tf'`：依詞頻排序，各取 TOP_K 個看漲/看跌鑑別詞，輸出 `bull_terms.csv` / `bear_terms.csv`
  - `'chi2'`：依卡方值排序（手動計算 bull/bear DF 偏向），輸出同上
  - `'sklearn'`：直接用 sklearn 的 `SelectKBest(chi2)` 篩出 TOP_K*2 個總特徵

In [ ]:
# ── Step 3-1：斷詞與每日合併（耗時，只需執行一次） ──
import csv
from datetime import datetime

# 讀入已貼標文章，排除中性（label=0）
articles = []
with open(PATH_ARTICLES_LABELED, newline='', encoding='utf-8') as f:
    for row in csv.DictReader(f):
        row['label'] = int(row['label'])
        articles.append(row)

bull_arts = [a for a in articles if a['label'] ==  1]
bear_arts = [a for a in articles if a['label'] == -1]
print(f'看漲文章：{len(bull_arts)} 篇')
print(f'看跌文章：{len(bear_arts)} 篇')

# ── 逐篇斷詞（保留 post_date 供後續每天合併）──
print(f'\n斷詞中（method={TOKENIZE_METHOD}）...')
_corpus_raw   = []  # 每篇的 tokens_str
_labels_raw   = []  # 每篇 label
_dates_raw    = []  # 每篇 post_date

for i, art in enumerate(articles):
    if i % 200 == 0:
        print(f'  {i}/{len(articles)}')
    if art['label'] == 0:
        continue

    post_date = datetime.strptime(art['post_time'], '%Y-%m-%d %H:%M:%S').date()
    text = (art.get('title') or '') + ' ' + (art.get('content') or '')
    tokens = tokenize(text)
    _corpus_raw.append(' '.join(tokens))
    _labels_raw.append(art['label'])
    _dates_raw.append(post_date)

print(f'\n斷詞完成，共 {len(_corpus_raw)} 篇（排除中性）')

# ── 每天合併成一筆 ──
_day_tokens  = {}   # date -> [tokens_str, ...]
_day_label   = {}   # date -> label

for d, t, l in zip(_dates_raw, _corpus_raw, _labels_raw):
    if d not in _day_tokens:
        _day_tokens[d]  = []
        _day_label[d]   = l
    _day_tokens[d].append(t)

daily_dates = sorted(_day_tokens.keys())
corpus  = [' '.join(_day_tokens[d]) for d in daily_dates]   # 每天一筆
labels  = [_day_label[d]            for d in daily_dates]

print(f'每天合併後：{len(corpus)} 天（原 {len(_corpus_raw)} 篇文章）')
print(f'看漲天數：{labels.count(1)}，看跌天數：{labels.count(-1)}')

# 儲存（每天一列）
with open(PATH_ARTICLES_PROCESSED, 'w', newline='', encoding='utf-8-sig') as f:
    writer = csv.writer(f)
    writer.writerow(['date', 'processed_text', 'label'])
    for d, text, label in zip(daily_dates, corpus, labels):
        writer.writerow([d, text, label])
print(f'已儲存至 {PATH_ARTICLES_PROCESSED}')

In [3]:
# ── Step 3-2：讀取已斷詞資料並切分訓練/測試集（極速，可隨意改參數重跑） ──
import csv
import numpy as np
from sklearn.model_selection import train_test_split as _tts

# 1. 從存好的 CSV 讀取資料，保證資料乾淨且不用重跑斷詞！
corpus_loaded = []
labels_loaded = []
daily_dates_loaded = []

print(f'正在載入已斷詞文本 {PATH_ARTICLES_PROCESSED} ...')
with open(PATH_ARTICLES_PROCESSED, newline='', encoding='utf-8-sig') as f:
    reader = csv.DictReader(f)
    for row in reader:
        daily_dates_loaded.append(row['date'])
        corpus_loaded.append(row['processed_text'])
        labels_loaded.append(int(row['label']))

n = len(corpus_loaded)
print(f'載入完成！共 {n} 天的資料準備進行切分。')

# 2. 依照參數進行訓練集/測試集切分
if SPLIT_METHOD == 'time':
    split_idx     = int(n * (1 - TEST_RATIO))
    train_indices = list(range(split_idx))
    test_indices  = list(range(split_idx, n))
    print(f'\n切割方式：時序（前 {int((1-TEST_RATIO)*100)}% 訓練 / 後 {int(TEST_RATIO*100)}% 測試）')
elif SPLIT_METHOD == 'random':
    all_idx = list(range(n))
    train_indices, test_indices = _tts(
        all_idx, test_size=TEST_RATIO, random_state=RANDOM_SEED, stratify=labels_loaded
    )
    print(f'\n切割方式：隨機 stratified（test_size={TEST_RATIO}, seed={RANDOM_SEED}）')
else:
    raise ValueError(f'未知的 SPLIT_METHOD: {SPLIT_METHOD}')

# 3. 產出最終的 Train / Test 變數
corpus_train = [corpus_loaded[i] for i in train_indices]
corpus_test  = [corpus_loaded[i] for i in test_indices]
y_train      = np.array([labels_loaded[i] for i in train_indices])
y_test       = np.array([labels_loaded[i] for i in test_indices])

print(f'訓練集：{len(corpus_train)} 天（漲={int(sum(y_train == 1))}, 跌={int(sum(y_train == -1))}）')
print(f'測試集：{len(corpus_test)} 天（漲={int(sum(y_test  == 1))}, 跌={int(sum(y_test  == -1))}）')

正在載入已斷詞文本 articles_processed.csv ...
載入完成！共 185 天的資料準備進行切分。

切割方式：隨機 stratified（test_size=0.2, seed=42）
訓練集：148 天（漲=82, 跌=66）
測試集：37 天（漲=21, 跌=16）


In [4]:
# ── 詞彙篩選：依 TERM_SELECT_METHOD 分流 ──
# 注意：TF-IDF 向量器統一加入 NGRAM_RANGE，
# 讓 monpa 斷出的「詞」再組成 bigram / trigram 詞組合（如「外資 買超」）

def compute_tf_df(corpus_list):
    """計算 TF（總詞頻）與 DF（文件頻率）。僅用於 'tf' 模式（unigram）。"""
    tf, df = Counter(), Counter()
    for text in corpus_list:
        grams = text.split()
        tf.update(grams)
        df.update(set(grams))
    return tf, df


def save_terms_csv(path, words, extra_cols_fn, header):
    with open(path, 'w', newline='', encoding='utf-8-sig') as f:
        writer = csv.writer(f)
        writer.writerow(header)
        for w in words:
            writer.writerow([w] + extra_cols_fn(w))


bull_selected, bear_selected = None, None

if TERM_SELECT_METHOD in ('tf', 'chi2'):
    bull_corpus = [c for c, l in zip(corpus, labels) if l ==  1]
    bear_corpus = [c for c, l in zip(corpus, labels) if l == -1]
    bull_tf, bull_df = compute_tf_df(bull_corpus)
    bear_tf, bear_df = compute_tf_df(bear_corpus)

    if TERM_SELECT_METHOD == 'tf':
        # [METHOD] 依 TF 詞頻排序（unigram 層級）
        # 注意：向量化時仍會套用 NGRAM_RANGE，但 vocabulary 只含 unigram
        bull_selected = [w for w, _ in bull_tf.most_common(TOP_K_WORDS)]
        bear_selected = [w for w, _ in bear_tf.most_common(TOP_K_WORDS)]

        save_terms_csv(PATH_BULL_TERMS, bull_selected,
                       lambda w: [bull_tf[w], bull_df[w]], ['Word', 'TF', 'DF'])
        save_terms_csv(PATH_BEAR_TERMS, bear_selected,
                       lambda w: [bear_tf[w], bear_df[w]], ['Word', 'TF', 'DF'])
        print(f'已依 TF 排序輸出 {PATH_BULL_TERMS} / {PATH_BEAR_TERMS}')

    elif TERM_SELECT_METHOD == 'chi2':
        # [METHOD] 依 sklearn chi2 + bull/bear DF 偏向拆分
        # 用 NGRAM_RANGE 建 TF 矩陣，chi2 分數同時涵蓋詞與詞組合
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.feature_selection import chi2

        vec_full = TfidfVectorizer(use_idf=False, lowercase=False,
                                   token_pattern=r'\S+', ngram_range=NGRAM_RANGE)
        X_full   = vec_full.fit_transform(corpus)
        vocab    = vec_full.get_feature_names_out()

        y_binary = np.array([1 if l == 1 else 0 for l in labels])
        chi2_scores, chi2_pval = chi2(X_full, y_binary)
        word_chi2 = dict(zip(vocab, chi2_scores))
        word_pval = dict(zip(vocab, chi2_pval))

        # 以 unigram DF 判斷主屬類別；N-gram 詞組依 chi2 值排序
        bull_selected = sorted(
            [w for w in vocab if bull_df.get(w.split()[0], 0) >  bear_df.get(w.split()[0], 0)],
            key=lambda w: word_chi2[w], reverse=True
        )[:TOP_K_WORDS]
        bear_selected = sorted(
            [w for w in vocab if bear_df.get(w.split()[0], 0) >= bull_df.get(w.split()[0], 0)],
            key=lambda w: word_chi2[w], reverse=True
        )[:TOP_K_WORDS]

        for path, words in [(PATH_BULL_TERMS, bull_selected),
                             (PATH_BEAR_TERMS, bear_selected)]:
            is_bull = (path == PATH_BULL_TERMS)
            tf_dict = bull_tf if is_bull else bear_tf
            df_dict = bull_df if is_bull else bear_df
            save_terms_csv(
                path, words,
                lambda w: [round(word_chi2[w], 4), round(word_pval[w], 6),
                           tf_dict.get(w.split()[0], 0), df_dict.get(w.split()[0], 0)],
                ['Word', 'Chi2', 'P-value', 'TF(head)', 'DF(head)'],
            )
        print(f'已依 Chi2 排序輸出 {PATH_BULL_TERMS} / {PATH_BEAR_TERMS}')

    print(f'看漲詞：{len(bull_selected)} 個，看跌詞：{len(bear_selected)} 個')

else:
    print(f'TERM_SELECT_METHOD={TERM_SELECT_METHOD}，詞表由 sklearn SelectKBest 於下一格處理')

TERM_SELECT_METHOD=sklearn，詞表由 sklearn SelectKBest 於下一格處理


In [5]:
# ── 向量化（統一套用 NGRAM_RANGE：詞級 N-gram）──
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2

if TERM_SELECT_METHOD in ('tf', 'chi2'):
    # 以 bull_selected ∪ bear_selected 作為固定 vocabulary（詞表選取維持原樣）
    termset = {}
    for w in list(bull_selected) + list(bear_selected):
        if w not in termset:
            termset[w] = len(termset)

    vectorizer = TfidfVectorizer(vocabulary=termset, use_idf=True, lowercase=False,
                                 token_pattern=r'\S+', ngram_range=NGRAM_RANGE,
                                 sublinear_tf=True)
    X = vectorizer.fit_transform(corpus)
    feature_names = np.array(list(termset.keys()))
    print(f'方法={TERM_SELECT_METHOD}：vocabulary 固定為 {X.shape[1]} 個詞（含 N-gram 詞組）')

    # 依切分索引拆出訓練 / 測試矩陣（詞表本身維持原樣，僅切分資料）
    X_train = X[train_indices]
    X_test  = X[test_indices]

elif TERM_SELECT_METHOD == 'sklearn':
    # ── Vectorizer 只對訓練集 fit，transform 兩者（防止 data leakage）──
    vectorizer = TfidfVectorizer(use_idf=True, lowercase=False,
                                 token_pattern=r'\S+', ngram_range=NGRAM_RANGE,
                                 sublinear_tf=True, min_df=2)
    X_full_train = vectorizer.fit_transform(corpus_train)
    X_full_test  = vectorizer.transform(corpus_test)
    print(f'原始矩陣（訓練集）：{X_full_train.shape}（含 {NGRAM_RANGE[0]}~{NGRAM_RANGE[1]}-gram 詞組）')

    # ── Chi-square selector 只對訓練集 fit，transform 兩者（防止 data leakage）──
    selector = SelectKBest(chi2, k=TOP_K_WORDS * 2)
    X_train = selector.fit_transform(X_full_train, y_train)
    X_test  = selector.transform(X_full_test)
    feature_names = vectorizer.get_feature_names_out()[selector.get_support()]
    print(f'選後訓練集：{X_train.shape}')
    print(f'選後測試集：{X_test.shape}')
    print(f'選出特徵（前 20）：{feature_names[:20].tolist()}')

    # ── 關鍵詞分析：計算各詞在看漲 / 看跌訓練樣本的平均 TF-IDF ──
    bull_mask = (y_train == 1)
    bear_mask = (y_train == -1)

    X_train_arr  = X_train.toarray()
    bull_mean    = X_train_arr[bull_mask].mean(axis=0)   # shape: (n_features,)
    bear_mean    = X_train_arr[bear_mask].mean(axis=0)
    diff         = bull_mean - bear_mean                  # 正 = 看漲偏向，負 = 看跌偏向

    direction = ['看漲' if d > 0 else '看跌' for d in diff]
    order     = np.argsort(-np.abs(diff))                 # 依絕對差值由大到小

    with open(PATH_KEYWORDS_ANALYSIS, 'w', newline='', encoding='utf-8-sig') as f:
        writer = csv.writer(f)
        writer.writerow(['詞', '看漲平均TF-IDF', '看跌平均TF-IDF', '差值', '方向'])
        for i in order:
            writer.writerow([
                feature_names[i],
                round(float(bull_mean[i]), 6),
                round(float(bear_mean[i]), 6),
                round(float(diff[i]),      6),
                direction[i],
            ])

    n_bull_kw = sum(d > 0 for d in diff)
    n_bear_kw = sum(d < 0 for d in diff)
    print(f'\n關鍵詞分析已儲存至 {PATH_KEYWORDS_ANALYSIS}')
    print(f'看漲偏向詞：{n_bull_kw} 個，看跌偏向詞：{n_bear_kw} 個')

else:
    raise ValueError(f'未知的 TERM_SELECT_METHOD: {TERM_SELECT_METHOD}')

# 儲存向量矩陣、標籤與特徵名稱
import pickle
sparse.save_npz(PATH_X_TRAIN, X_train)
sparse.save_npz(PATH_X_TEST,  X_test)
np.save(PATH_Y_TRAIN, y_train)
np.save(PATH_Y_TEST,  y_test)
sparse.save_npz(PATH_VECTOR_NPZ, X_train)  # 保留相容性（Step 5 不讀此檔）
with open(PATH_FEATURE_NAMES, 'wb') as f:
    pickle.dump(list(feature_names), f)

print(f'\n訓練集矩陣已儲存至 {PATH_X_TRAIN}（{X_train.shape}）')
print(f'測試集矩陣已儲存至 {PATH_X_TEST}（{X_test.shape}）')
print(f'特徵詞彙已儲存至  {PATH_FEATURE_NAMES}')

原始矩陣（訓練集）：(148, 24651)（含 1~2-gram 詞組）
選後訓練集：(148, 400)
選後測試集：(37, 400)
選出特徵（前 20）：['三角形', '上市櫃 公司', '上方', '上海商銀', '上車', '下挫 影響', '下挫 跌幅', '下跌 影響', '下車', '不用', '中信金 聯電', '中期', '中興電 營收', '中興電 開關', '中鋼 中鴻', '中電', '主因', '亞力 重電', '交易 學習', '交易所']

關鍵詞分析已儲存至 keywords_analysis.csv
看漲偏向詞：100 個，看跌偏向詞：300 個

訓練集矩陣已儲存至 X_train.npz（(148, 400)）
測試集矩陣已儲存至 X_test.npz（(37, 400)）
特徵詞彙已儲存至  feature_names.pkl


In [6]:
# ── 預覽向量矩陣（訓練集前 5 列 × 前 10 欄）──
df_preview = pd.DataFrame(
    X_train[:5, :10].toarray(),
    columns=feature_names[:10],
)
df_preview

,三角形,上市櫃 公司,上方,上海商銀,上車,下挫 影響,下挫 跌幅,下跌 影響,下車,不用
0,0.000000,0.0,0.0,0.0,0.134666,0.0,0.0,0.0,0.0,0.000000
1,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000
2,0.172657,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000
3,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.039208
4,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000


---
# Step 4 — 分類器訓練與評估

## 切割方式切換（`SPLIT_METHOD`）

- `'time'`：依時序切（前 80% 訓練 / 後 20% 測試），**符合金融場景，避免未來資料洩漏**
- `'random'`：`train_test_split` + `stratify`，標籤分布均衡但會洩漏時序資訊

## 分類器

NB / KNN / SVM / Decision Tree / Random Forest

In [7]:
# ── Step 4 —（切分已於 Step 3 完成，直接載入）──
# 若需調整切分方式，請至 Step 3 的 s3-tokenize cell 修改 SPLIT_METHOD 參數後重新執行 Step 3

X_train = sparse.load_npz(PATH_X_TRAIN)
X_test  = sparse.load_npz(PATH_X_TEST)
y_train = np.load(PATH_Y_TRAIN)
y_test  = np.load(PATH_Y_TEST)

print(f'訓練集：{X_train.shape[0]} 天，{X_train.shape[1]} 維特徵')
print(f'測試集：{X_test.shape[0]} 天，{X_test.shape[1]} 維特徵')
print(f'訓練集標籤 — 看漲：{sum(y_train == 1)} 天，看跌：{sum(y_train == -1)} 天')
print(f'測試集標籤 — 看漲：{sum(y_test  == 1)} 天，看跌：{sum(y_test  == -1)} 天')

訓練集：148 天，400 維特徵
測試集：37 天，400 維特徵
訓練集標籤 — 看漲：82 天，看跌：66 天
測試集標籤 — 看漲：21 天，看跌：16 天


In [12]:
# ── 評估函式 ──
from sklearn.naive_bayes    import ComplementNB
from sklearn.neighbors      import KNeighborsClassifier
from sklearn.svm            import SVC
from sklearn.tree           import DecisionTreeClassifier
from sklearn.ensemble       import RandomForestClassifier
from sklearn.linear_model   import LogisticRegression
from sklearn.metrics        import accuracy_score, confusion_matrix, classification_report
import lightgbm as lgb
import xgboost  as xgb


def evaluate(name, model, X_tr, y_tr, X_te, y_te):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    acc = accuracy_score(y_te, y_pred)
    cm  = confusion_matrix(y_te, y_pred, labels=[1, -1])

    print(f'\n===== {name} =====')
    print(f'準確率：{acc * 100:.1f}%\n')
    print('Classification Report:')
    print(classification_report(y_te, y_pred, labels=[1, -1],
                                target_names=['看漲', '看跌']))
    print('Confusion Matrix（列=真實，欄=預測）：')
    print(f'{"":10s}預測漲  預測跌')
    print(f'真實漲    {cm[0][0]:5d}  {cm[0][1]:5d}')
    print(f'真實跌    {cm[1][0]:5d}  {cm[1][1]:5d}')
    return acc

In [ ]:
# # 將標籤從 [-1, 1] 轉換為 [0, 1]
# # 如果你的 y 本來就是字串（如 '看漲', '看跌'），這步也會自動幫你轉成 0 和 1
# from sklearn.preprocessing import LabelEncoder

# le = LabelEncoder()
# y_train = le.fit_transform(y_train)
# y_test = le.transform(y_test)

# # 這樣之後再執行你的 models 迴圈就不會報錯了

In [13]:
# ── 訓練 8 種分類器 ──
models = [
    ('Complement NB',       ComplementNB()),
    ('KNN (k=5)',           KNeighborsClassifier(n_neighbors=5)),
    ('SVM',                 SVC(kernel='linear', class_weight='balanced')),
    ('Decision Tree',       DecisionTreeClassifier(random_state=RANDOM_SEED, class_weight='balanced')),
    ('Random Forest',       RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED, class_weight='balanced')),
    ('Logistic Regression', LogisticRegression(C=1.0, class_weight='balanced', max_iter=2000)),
    ('LightGBM',            lgb.LGBMClassifier(n_estimators=200, class_weight='balanced',
                                random_state=RANDOM_SEED, verbose=-1)),
    ('XGBoost',             xgb.XGBClassifier(n_estimators=200, scale_pos_weight=1.24,
                                random_state=RANDOM_SEED, eval_metric='logloss', verbosity=0)),
]

results = []
for name, model in models:
    acc = evaluate(name, model, X_train, y_train, X_test, y_test)
    results.append((name, acc))

print('\n' + '=' * 40)
print('準確率彙整')
print('=' * 40)
for name, acc in sorted(results, key=lambda x: x[1], reverse=True):
    print(f'  {name:20s}: {acc * 100:.1f}%')


===== Complement NB =====
準確率：45.9%

Classification Report:
              precision    recall  f1-score   support

          看漲       0.52      0.52      0.52        21
          看跌       0.00      0.00      0.00         0

   micro avg       0.52      0.52      0.52        21
   macro avg       0.26      0.26      0.26        21
weighted avg       0.52      0.52      0.52        21

Confusion Matrix（列=真實，欄=預測）：
          預測漲  預測跌
真實漲       11      0
真實跌        0      0

===== KNN (k=5) =====
準確率：56.8%

Classification Report:
              precision    recall  f1-score   support

          看漲       0.57      1.00      0.72        21
          看跌       0.00      0.00      0.00         0

   micro avg       0.57      1.00      0.72        21
   macro avg       0.28      0.50      0.36        21
weighted avg       0.57      1.00      0.72        21

Confusion Matrix（列=真實，欄=預測）：
          預測漲  預測跌
真實漲       21      0
真實跌        0      0

===== SVM =====
準確率：45.9%

Classification Report:
 

c:\Users\a3504\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\a3504\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\a3504\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\a3504\anaconda3\Lib\site-package


===== LightGBM =====
準確率：43.2%

Classification Report:
              precision    recall  f1-score   support

          看漲       0.50      0.33      0.40        21
          看跌       0.00      0.00      0.00         0

   micro avg       0.50      0.33      0.40        21
   macro avg       0.25      0.17      0.20        21
weighted avg       0.50      0.33      0.40        21

Confusion Matrix（列=真實，欄=預測）：
          預測漲  預測跌
真實漲        7      0
真實跌        0      0

===== XGBoost =====
準確率：45.9%

Classification Report:
              precision    recall  f1-score   support

          看漲       0.52      0.62      0.57        21
          看跌       0.00      0.00      0.00         0

   micro avg       0.52      0.62      0.57        21
   macro avg       0.26      0.31      0.28        21
weighted avg       0.52      0.62      0.57        21

Confusion Matrix（列=真實，欄=預測）：
          預測漲  預測跌
真實漲       13      0
真實跌        0      0

準確率彙整
  KNN (k=5)           : 56.8%
  Logistic Regression 

c:\Users\a3504\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\a3504\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\a3504\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\a3504\anaconda3\Lib\site-package

---
# Step 5 — 移動回測

模擬真實預測場景：逐日往後推，每日以**前 `WINDOW_DAYS` 個日曆天**的文章訓練，預測當日文章多數方向。

## 三層出手門檻

1. 當日有效文章（排除中性）≥ `MIN_ARTICLES`
2. 訓練集須同時包含看漲與看跌樣本
3. 預測結果的多空差距 / 總數 ≥ `SIGNAL_RATIO`（信號要夠強）

## 向量化方式

為與 Step 3 保持一致，Step 5 使用**相同的 `feature_names` 與 `TOKENIZE_METHOD`**重新組裝訓練/測試向量。

In [76]:
# ── Step 5 參數覆寫 ──
# 例：WINDOW_DAYS = 60

import pickle
import csv
from datetime import datetime
from collections import defaultdict

# 1. 載入 feature_names（若 Step 3 已執行則直接取用，否則從 pkl 讀入）
if 'feature_names' not in vars():
    with open(PATH_FEATURE_NAMES, 'rb') as f:
        feature_names = np.array(pickle.load(f))
    print(f'從 {PATH_FEATURE_NAMES} 載入 feature_names（{len(feature_names)} 個）')

# 2. 極速計算 day_counts：只讀標籤檔算篇數，絕對不再斷詞！
day_counts = defaultdict(int)
all_dates_set = set()

print('正在掃描原始文章計算每日篇數...')
with open(PATH_ARTICLES_LABELED, newline='', encoding='utf-8') as f:
    for row in csv.DictReader(f):
        try:
            d = datetime.strptime(row['post_time'], '%Y-%m-%d %H:%M:%S').date()
            day_counts[d] += 1
            all_dates_set.add(d)
        except Exception:
            pass

all_dates = sorted(list(all_dates_set))
print(f'共 {len(all_dates)} 個日期')

# 3. 從 articles_processed.csv 讀取「已經斷好詞」的文本與標籤
print('直接載入已斷詞的文本 (articles_processed.csv)...')
day_texts  = {}   # date -> 當日所有文章合併後的 tokens_str
day_labels = {}   # date -> label（只記錄非中性日）

with open(PATH_ARTICLES_PROCESSED, newline='', encoding='utf-8-sig') as f:
    for row in csv.DictReader(f):
        d = datetime.strptime(row['date'], '%Y-%m-%d').date()
        day_texts[d] = row['processed_text']
        day_labels[d] = int(row['label'])

labeled_dates = sorted(list(day_texts.keys()))

print(f'載入完成。有標籤日期：{len(labeled_dates)} 天')
print(f'使用 TOKENIZE_METHOD={TOKENIZE_METHOD}，NGRAM_RANGE={NGRAM_RANGE}')

正在掃描原始文章計算每日篇數...
共 321 個日期
直接載入已斷詞的文本 (articles_processed.csv)...
載入完成。有標籤日期：185 天
使用 TOKENIZE_METHOD=monpa，NGRAM_RANGE=(1, 2)


In [77]:
# ── 向量化函式（per-fold 動態選詞，防止 leakage）──
from sklearn.feature_extraction.text import TfidfVectorizer as _TF
from sklearn.feature_selection import SelectKBest, chi2 as _chi2

def vectorize_fold(train_texts: list, test_texts: list, train_y, k: int = 100):
    """
    每個 fold 獨立 fit vectorizer + selector，只看訓練文本。
    - TfidfVectorizer: ngram_range=NGRAM_RANGE, min_df=2, sublinear_tf=True
    - SelectKBest(chi2, k=k)：只對訓練集 fit，test 僅 transform
    - 若訓練集特徵數不足 k，自動縮減至實際特徵數
    回傳 (X_tr, X_te)
    """
    vec = _TF(lowercase=False, token_pattern=r'\S+',
              ngram_range=NGRAM_RANGE, min_df=2, sublinear_tf=True)
    X_tr_full = vec.fit_transform(train_texts)
    X_te_full = vec.transform(test_texts)

    actual_k = min(k, X_tr_full.shape[1])
    sel = SelectKBest(_chi2, k=actual_k)
    X_tr = sel.fit_transform(X_tr_full, train_y)
    X_te = sel.transform(X_te_full)
    return X_tr, X_te

In [78]:
# ── 移動回測主迴圈（每天一筆，使用每日合併文本）──
from sklearn.ensemble import VotingClassifier, RandomForestClassifier
from sklearn.tree     import DecisionTreeClassifier
from sklearn.naive_bayes import ComplementNB
from sklearn.metrics  import accuracy_score, confusion_matrix

results_bt = []   # (date, pred_dir, actual_dir)

for test_date in labeled_dates:
    # ── 訓練窗口：前 WINDOW_DAYS 個日曆天，只取有標籤的日期 ──
    window_start = test_date - timedelta(days=WINDOW_DAYS)
    train_dates  = [d for d in labeled_dates if window_start <= d < test_date]
    train_texts  = [day_texts[d] for d in train_dates]
    train_y      = [day_labels[d] for d in train_dates]

    # ── 門檻 1：當日文章篇數不足 ──
    if day_counts.get(test_date, 0) < MIN_ARTICLES:
        continue

    # ── 門檻 2：訓練集需同時含看漲/看跌 ──
    if len(train_texts) < 2 or len(set(train_y)) < 2:
        continue

    # ── per-fold 動態選詞向量化（只 fit 訓練集）──
    X_tr, X_te = vectorize_fold(train_texts, [day_texts[test_date]], train_y)

    # ── VotingClassifier（hard voting）──
    model = VotingClassifier(estimators=[
        ('dt',  DecisionTreeClassifier(class_weight='balanced')),
        ('rf',  RandomForestClassifier(n_estimators=100, class_weight='balanced_subsample')),
        ('cnb', ComplementNB()),
    ], voting='hard')
    model.fit(X_tr, train_y)
    pred_dir   = int(model.predict(X_te)[0])
    actual_dir = day_labels[test_date]

    results_bt.append((test_date, pred_dir, actual_dir))

print(f'回測完成，出手 {len(results_bt)} 次（共 {len(labeled_dates)} 個有標籤日）')

回測完成，出手 128 次（共 185 個有標籤日）


In [79]:
# ── 回測結果彙整 ──
n_total = len(all_dates)
n_trade = len(results_bt)
trade_rate = n_trade / n_total * 100 if n_total else 0

print('=' * 40)
print('移動回測結果')
print('=' * 40)
print(f'總文章天數：{n_total} 天')
print(f'出手天數：  {n_trade} 天（出手率：{trade_rate:.1f}%）')

if n_trade > 0:
    y_actual    = [r[2] for r in results_bt]
    y_predicted = [r[1] for r in results_bt]
    cm  = confusion_matrix(y_actual, y_predicted, labels=[1, -1])
    acc = accuracy_score(y_actual, y_predicted)

    print('\nConfusion Matrix：')
    print(f'{"":10s}預測漲  預測跌')
    print(f'真實漲    {cm[0][0]:5d}  {cm[0][1]:5d}')
    print(f'真實跌    {cm[1][0]:5d}  {cm[1][1]:5d}')
    print(f'\n總準確率：{acc * 100:.1f}%')
else:
    print('（沒有符合出手條件的日期）')

移動回測結果
總文章天數：321 天
出手天數：  128 天（出手率：39.9%）

Confusion Matrix：
          預測漲  預測跌
真實漲       45     24
真實跌       31     28

總準確率：57.0%


In [47]:
# ── 逐日預測明細（前 20 筆）──
print(f'{"日期":12s}  預測  實際  結果')
print('-' * 35)
for date, pred, actual in results_bt[:20]:
    pred_s   = '漲' if pred   ==  1 else '跌'
    actual_s = '漲' if actual ==  1 else '跌'
    ok       = '✓' if pred == actual else '✗'
    print(f'{str(date):12s}  {pred_s}     {actual_s}     {ok}')

日期            預測  實際  結果
-----------------------------------
2023-03-09    跌     跌     ✓
2023-03-10    跌     跌     ✓
2023-04-06    漲     漲     ✓
2023-04-07    漲     漲     ✓
2023-04-09    漲     漲     ✓
2023-04-22    漲     跌     ✗
2023-04-24    漲     跌     ✗
2023-04-28    漲     漲     ✓
2023-05-01    漲     跌     ✗
2023-05-02    漲     跌     ✗
2023-05-03    漲     漲     ✓
2023-05-04    漲     漲     ✓
2023-05-08    漲     跌     ✗
2023-05-10    漲     漲     ✓
2023-05-12    漲     漲     ✓
2023-05-15    漲     漲     ✓
2023-05-16    漲     漲     ✓
2023-05-17    漲     漲     ✓
2023-05-18    漲     漲     ✓
2023-05-19    漲     漲     ✓
